In [ ]:
import pandas as pd
import pyomo.environ as pyo
from pyomo.opt import SolverFactory

# Loading the data

def load_ties_data():
    """Load ties dataset (no common quotas)."""
    apps_df = pd.read_csv('ties_applications.csv')
    colleges_df = pd.read_csv('ties_colleges.csv')
    students_df = pd.read_csv('ties_students.csv')

    students = students_df['student_id'].tolist()
    colleges = colleges_df['college_id'].tolist()
    college_cap = colleges_df.set_index('college_id')['capacity'].to_dict()

    apps = {}
    for _, row in apps_df.iterrows():
        apps[(row['student_id'], row['college_id'])] = {
            'rank': row['rank'],
            'score': row['score']
        }

    return students, colleges, college_cap, apps


# Chilean ties – continuous cutoff (SO-C-NW-CUT)


def build_model(students, colleges, college_cap, apps):
    model = pyo.ConcreteModel()

    # Sets
    model.I = pyo.Set(initialize=students)
    model.J = pyo.Set(initialize=colleges)
    model.E = pyo.Set(initialize=apps.keys())

    # Parameters
    model.u = pyo.Param(model.J, initialize=college_cap)
    model.rank = pyo.Param(model.E, initialize={(i,j): apps[(i,j)]['rank'] for (i,j) in apps})
    model.score = pyo.Param(model.E, initialize={(i,j): apps[(i,j)]['score'] for (i,j) in apps})
    max_rank = max(model.rank[i,j] for (i,j) in model.E)
    model.K = pyo.Param(initialize=max_rank + 1)
    max_score = max(model.score[i,j] for (i,j) in model.E)
    model.M = pyo.Param(initialize=max_score + 1, doc='Big-M')
    eps = 0.001
    model.eps = pyo.Param(initialize=eps)

    # Variables
    model.x = pyo.Var(model.E, within=pyo.Binary)  # assignment
    model.t = pyo.Var(model.J, within=pyo.NonNegativeReals, bounds=(0, max_score))  # cutoff
    model.f = pyo.Var(model.J, within=pyo.Binary)  # college full indicator
    model.dbar = pyo.Var(model.E, within=pyo.Binary)  # marginal students

    # (1) Student limit: each student assigned to at most one college
    def student_limit_rule(m, i):
        return sum(m.x[i,j] for j in model.J if (i,j) in model.E) <= 1
    model.student_limit = pyo.Constraint(model.I, rule=student_limit_rule)

    # (5) If x=1 then score >= cutoff
    def cutoff_met_rule(m, i, j):
        return m.t[j] <= (1 - m.x[i,j]) * m.M + m.score[i,j]
    model.cutoff_met = pyo.Constraint(model.E, rule=cutoff_met_rule)

    # (6) Envy-freeness: if not assigned to better/equal, then score + eps <= cutoff
    def envy_free_rule(m, i, j):
        better_or_equal = [h for h in model.J if (i,h) in model.E and m.rank[i,h] <= m.rank[i,j]]
        sum_better = sum(m.x[i,h] for h in better_or_equal)
        return m.score[i,j] + m.eps <= m.t[j] + (sum_better) * m.M
    model.envy_free = pyo.Constraint(model.E, rule=envy_free_rule)

    # (7) If f=1 then college is full (sum x >= u)
    def full_if_f_rule(m, j):
        return m.f[j] * m.u[j] <= sum(m.x[i,j] for i in model.I if (i,j) in model.E)
    model.full_if_f = pyo.Constraint(model.J, rule=full_if_f_rule)

    # (8) If cutoff > 0 then f=1 (t <= f * M)
    def t_f_rule(m, j):
        return m.t[j] <= m.f[j] * m.M
    model.t_f = pyo.Constraint(model.J, rule=t_f_rule)

    # (22) dbar <= x
    def dbar_limit_rule(m, i, j):
        return m.dbar[i,j] <= m.x[i,j]
    model.dbar_limit = pyo.Constraint(model.E, rule=dbar_limit_rule)

    # (23) dbar=1 implies score equals cutoff
    def dbar_ge_rule(m, i, j):
        return (m.dbar[i,j] - 1) * m.M + m.score[i,j] <= m.t[j]
    model.dbar_ge = pyo.Constraint(model.E, rule=dbar_ge_rule)

    def dbar_le_rule(m, i, j):
        return m.t[j] <= m.score[i,j] + (1 - m.dbar[i,j]) * m.M
    model.dbar_le = pyo.Constraint(model.E, rule=dbar_le_rule)

    # (24) Non-wastefulness
    def non_waste_rule(m, j):
        return sum(m.x[i,j] - m.dbar[i,j] for i in model.I if (i,j) in model.E) <= m.u[j] - 1
    model.non_waste = pyo.Constraint(model.J, rule=non_waste_rule)

    # Objective (10): Student-optimal 
    def objective_rule(m):
        return sum((m.K - m.rank[i,j]) * m.x[i,j] for (i,j) in model.E)
    model.obj = pyo.Objective(rule=objective_rule, sense=pyo.maximize)

    return model


# Solve and output


def solve_model(model, solver_name='cplex'):
    solver = SolverFactory(solver_name)
    result = solver.solve(model, tee=True)
    return result

def print_results(model, name):
    print(f"\n=== {name} ===")
    try:
        obj_value = pyo.value(model.obj)
        print("Objective value:", obj_value)
    except (AttributeError, ValueError):
        print("No feasible solution found or model not solved.")
        return
    
    assigned = [(i,j) for (i,j) in model.E if pyo.value(model.x[i,j]) > 0.5]
    print(f"Number of assigned students: {len(assigned)}")
    
    # Print cutoffs
    for j in model.J:
        t_val = pyo.value(model.t[j])
        if t_val is not None:
            print(f"College {j} cutoff: {t_val:.2f}")
        else:
            print(f"College {j} cutoff: None")


if __name__ == "__main__":
    students, colleges, college_cap, apps = load_ties_data()
    model = build_model(students, colleges, college_cap, apps)
    solve_model(model)
    print_results(model, "Chilean ties (continuous)")


Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer 22.1.0.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2022.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\AmirReza\AppData\Local\Temp\tmpm5u06a08.cplex.log' open.
CPLEX> Problem 'C:\Users\AmirReza\AppData\Local\Temp\tmp76z7__b2.pyomo.lp' read.
Read time = 0.08 sec. (5.44 ticks)
CPLEX> Problem name         : C:\Users\AmirReza\AppData\Local\Temp\tmp76z7__b2.pyomo.lp
Objective sense      : Maximize
Variables            :   22356  [Box: 20,  Binary: 22336]
Objective nonzeros   :   11158
Linear constraints   :   57350  [Less: 46192,  Greater: 11158]
  Nonzeros           :  194353
  RHS nonzeros       :   46152

Variables            : Min LB: 0.000000         Max UB: 500.0000       
Objective n

In [ ]:
import pandas as pd
import pyomo.environ as pyo
from pyomo.opt import SolverFactory

# Loading the data

def load_ties_data():
    """Load ties dataset (no common quotas)."""
    apps_df = pd.read_csv('ties_applications.csv')
    colleges_df = pd.read_csv('ties_colleges.csv')
    students_df = pd.read_csv('ties_students.csv')

    students = students_df['student_id'].tolist()
    colleges = colleges_df['college_id'].tolist()
    college_cap = colleges_df.set_index('college_id')['capacity'].to_dict()

    apps = {}
    for _, row in apps_df.iterrows():
        apps[(row['student_id'], row['college_id'])] = {
            'rank': row['rank'],
            'score': row['score']
        }

    return students, colleges, college_cap, apps

# Chilean ties – binary cutoff (SO-C-NW-BIN-CUT)

def build_model(students, colleges, college_cap, apps):
    model = pyo.ConcreteModel()

    # Sets
    model.I = pyo.Set(initialize=students)
    model.J = pyo.Set(initialize=colleges)
    model.E = pyo.Set(initialize=apps.keys())

    # Score levels per college
    score_levels = {}
    for j in colleges:
        scores = sorted({apps[(i,j)]['score'] for i in students if (i,j) in apps})
        score_levels[j] = scores
    model.score_levels = score_levels

    # Set of (college, level) pairs
    model.JK = pyo.Set(initialize=[(j,k) for j in colleges for k in range(1, len(score_levels[j])+1)])

    # Number of levels per college
    model.K = pyo.Set(model.J, initialize=lambda m, j: range(1, len(score_levels[j])+1))

    # Parameters
    model.u = pyo.Param(model.J, initialize=college_cap)
    model.rank = pyo.Param(model.E, initialize={(i,j): apps[(i,j)]['rank'] for (i,j) in apps})
    model.score = pyo.Param(model.E, initialize={(i,j): apps[(i,j)]['score'] for (i,j) in apps})

    def score_idx_rule(m, i, j):
        return score_levels[j].index(apps[(i,j)]['score']) + 1
    model.score_idx = pyo.Param(model.E, initialize=score_idx_rule)

    max_rank = max(model.rank[i,j] for (i,j) in model.E)
    model.K_const = pyo.Param(initialize=max_rank + 1)

    # Variables
    model.x = pyo.Var(model.E, within=pyo.Binary)
    model.t = pyo.Var(model.JK, within=pyo.Binary)
    model.dbar = pyo.Var(model.E, within=pyo.Binary)

    # (1) Student limit
    def student_limit_rule(m, i):
        return sum(m.x[i,j] for j in m.J if (i,j) in m.E) <= 1
    model.student_limit = pyo.Constraint(model.I, rule=student_limit_rule)

    # (11) Cutoff met: x <= t[score_idx]
    def cutoff_met_rule(m, i, j):
        return m.x[i,j] <= m.t[j, m.score_idx[i,j]]
    model.cutoff_met = pyo.Constraint(model.E, rule=cutoff_met_rule)

    # (12) Monotonicity: t[j,k] <= t[j,k+1]
    def monotonicity_rule(m, j, k):
        if k < len(score_levels[j]):
            return m.t[j,k] <= m.t[j,k+1]
        return pyo.Constraint.Skip
    model.monotonicity = pyo.Constraint(model.JK, rule=monotonicity_rule)

    # (13) Envy-freeness
    def envy_free_rule(m, i, j):
        better_or_equal = [h for h in m.J if (i,h) in m.E and m.rank[i,h] <= m.rank[i,j]]
        assigned_better = sum(m.x[i,h] for h in better_or_equal)
        return 1 <= assigned_better + (1 - m.t[j, m.score_idx[i,j]])
    model.envy_free = pyo.Constraint(model.E, rule=envy_free_rule)

    # (14) Equivalent: if cutoff > min score then college is full
    def non_min_cutoff_full_rule(m, j):
        return (1 - m.t[j,1]) * m.u[j] <= sum(m.x[i,j] for i in m.I if (i,j) in m.E)
    model.non_min_cutoff_full = pyo.Constraint(model.J, rule=non_min_cutoff_full_rule)

    # (22) dbar <= x
    def dbar_limit_rule(m, i, j):
        return m.dbar[i,j] <= m.x[i,j]
    model.dbar_limit = pyo.Constraint(model.E, rule=dbar_limit_rule)

    # (25) dbar identifies cutoff-level students
    def dbar_cutoff_rule(m, i, j):
        k = m.score_idx[i,j]
        if k > 1:
            return m.dbar[i,j] <= m.t[j,k] - m.t[j,k-1]
        else:
            return m.dbar[i,j] <= m.t[j,1]
    model.dbar_cutoff = pyo.Constraint(model.E, rule=dbar_cutoff_rule)

    # (24) Non-wastefulness:
    def non_waste_rule(m, j):
        return sum(m.x[i,j] - m.dbar[i,j] for i in m.I if (i,j) in m.E) <= m.u[j] - 1
    model.non_waste = pyo.Constraint(model.J, rule=non_waste_rule)

    # Objective (10)
    def objective_rule(m):
        return sum((m.K_const - m.rank[i,j]) * m.x[i,j] for (i,j) in m.E)
    model.obj = pyo.Objective(rule=objective_rule, sense=pyo.maximize)

    return model

# Solve and output

def solve_model(model, solver_name='cplex'):
    solver = SolverFactory(solver_name)
    result = solver.solve(model, tee=True)
    return result

def print_results(model, name):
    print(f"\n=== {name} ===")
    try:
        obj_value = pyo.value(model.obj)
        print("Objective value:", obj_value)
    except (AttributeError, ValueError):
        print("No feasible solution found or model not solved.")
        return
    
    assigned = [(i,j) for (i,j) in model.E if pyo.value(model.x[i,j]) > 0.5]
    print(f"Number of assigned students: {len(assigned)}")
    for j in model.J:
        cut = None
        for k in model.K[j]:
            if pyo.value(model.t[j,k]) > 0.5:
                cut = model.score_levels[j][k-1]
                break
        print(f"College {j} cutoff: {cut}")

if __name__ == "__main__":
    students, colleges, college_cap, apps = load_ties_data()
    model = build_model(students, colleges, college_cap, apps)
    solve_model(model)
    print_results(model, "Chilean ties (binary)")


Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer 22.1.0.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2022.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\AmirReza\AppData\Local\Temp\tmpk5rmnjon.cplex.log' open.
CPLEX> Problem 'C:\Users\AmirReza\AppData\Local\Temp\tmpdv_u9tub.pyomo.lp' read.
Read time = 0.06 sec. (4.98 ticks)
CPLEX> Problem name         : C:\Users\AmirReza\AppData\Local\Temp\tmpdv_u9tub.pyomo.lp
Objective sense      : Maximize
Variables            :   23327  [Binary: 23327]
Objective nonzeros   :   11158
Linear constraints   :   47163  [Less: 36005,  Greater: 11158]
  Nonzeros           :  184886
  RHS nonzeros       :    1540

Variables            : Min LB: 0.000000         Max UB: 1.000000       
Objective nonzeros   

In [ ]:
import pandas as pd
import pyomo.environ as pyo
from pyomo.opt import SolverFactory

# Loading the data

def load_strict_data():
    """Load strict dataset (with common quotas)."""
    apps_df = pd.read_csv('strict_applications.csv')
    colleges_df = pd.read_csv('strict_colleges.csv')
    groups_df = pd.read_csv('strict_groups.csv')
    students_df = pd.read_csv('strict_students.csv')

    students = students_df['student_id'].tolist()
    colleges = colleges_df['college_id'].tolist()
    college_cap = colleges_df.set_index('college_id')['capacity'].to_dict()
    college_subject = colleges_df.set_index('college_id')['subject_id'].to_dict()
    groups = groups_df['group_id'].tolist()
    group_cap = groups_df.set_index('group_id')['group_capacity'].to_dict()

    apps = {}
    for _, row in apps_df.iterrows():
        apps[(row['student_id'], row['college_id'])] = {
            'rank': row['rank'],
            'score': row['score'],
            'subject': row['subject_id']
        }

    return students, colleges, college_cap, college_subject, groups, group_cap, apps

# Common quotas – continuous cutoff (COM-SO-CUT)

def build_model(students, colleges, college_cap, college_subject, groups, group_cap, apps):
    model = pyo.ConcreteModel()

    # Sets
    model.I = pyo.Set(initialize=students)
    model.J = pyo.Set(initialize=colleges)
    model.E = pyo.Set(initialize=apps.keys())

    # Build groups: subject groups + singleton college groups
    group_colleges = {}
    for g in groups:
        group_colleges[g] = [j for j in colleges if college_subject[j] == g]
    for j in colleges:
        group_colleges[f'col_{j}'] = [j]

    model.G = pyo.Set(initialize=group_colleges.keys())

    group_cap_dict = {}
    for g in group_colleges:
        if isinstance(g, str) and g.startswith('col_'):
            j = int(g.split('_')[1])
            group_cap_dict[g] = college_cap[j]
        else:
            group_cap_dict[g] = group_cap[g]
    model.u_g = pyo.Param(model.G, initialize=group_cap_dict)

    # For each group, the set of applications
    group_apps = {}
    for g in model.G:
        cols = group_colleges[g]
        group_apps[g] = [(i,j) for (i,j) in apps if j in cols]
    model.G_apps = pyo.Set(model.G, initialize=group_apps)

    # Parameters
    model.rank = pyo.Param(model.E, initialize={(i,j): apps[(i,j)]['rank'] for (i,j) in apps})
    model.score = pyo.Param(model.E, initialize={(i,j): apps[(i,j)]['score'] for (i,j) in apps})
    max_rank = max(model.rank[i,j] for (i,j) in model.E)
    model.K = pyo.Param(initialize=max_rank + 1)
    max_score = max(model.score[i,j] for (i,j) in model.E)
    model.M = pyo.Param(initialize=max_score + 1)
    eps = 0.001
    model.eps = pyo.Param(initialize=eps)

    # Variables
    model.x = pyo.Var(model.E, within=pyo.Binary)
    model.t = pyo.Var(model.G, within=pyo.NonNegativeReals, bounds=(0, max_score))
    model.b = pyo.Var(model.J, model.G, within=pyo.Binary, initialize=0)
    model.f = pyo.Var(model.G, within=pyo.Binary)

    # (1) Student limit
    def student_limit_rule(m, i):
        return sum(m.x[i,j] for j in model.J if (i,j) in model.E) <= 1
    model.student_limit = pyo.Constraint(model.I, rule=student_limit_rule)

    # (26) Group quotas (common quotas + individual college capacities)
    def group_quota_rule(m, g):
        return sum(m.x[i,j] for (i,j) in m.G_apps[g]) <= m.u_g[g]
    model.group_quota = pyo.Constraint(model.G, rule=group_quota_rule)

    # (27) Cutoff met: for each group containing college j, t_g <= (1-x)*M + score
    for (i,j) in apps:
        for g in model.G:
            if j in group_colleges[g]:
                setattr(model, f'cutoff_met_{i}_{j}_{g}',
                        pyo.Constraint(expr=model.t[g] <= (1 - model.x[i,j]) * model.M + model.score[i,j]))

    # (28) Envy-freeness: score + eps <= t_g + (sum_better + (1 - b[j,g])) * M
    def envy_free_rule(m, i, j, g):
        if j not in group_colleges[g]:
            return pyo.Constraint.Skip
        better_or_equal = [h for h in model.J if (i,h) in model.E and m.rank[i,h] <= m.rank[i,j]]
        sum_better = sum(m.x[i,h] for h in better_or_equal)
        return m.score[i,j] + m.eps <= m.t[g] + (sum_better + (1 - m.b[j,g])) * m.M
    model.envy_free = pyo.Constraint(model.E, model.G, rule=envy_free_rule)

    # (29) For each college j, at least one group containing j has b[j,g]=1
    def b_sum_rule(m, j):
        groups_containing_j = [g for g in model.G if j in group_colleges[g]]
        return sum(m.b[j,g] for g in groups_containing_j) >= 1
    model.b_sum = pyo.Constraint(model.J, rule=b_sum_rule)

    # (30) If f_g=1 then group g is full (sum x >= u_g)
    def f_full_rule(m, g):
        return m.f[g] * m.u_g[g] <= sum(m.x[i,j] for (i,j) in m.G_apps[g])
    model.f_full = pyo.Constraint(model.G, rule=f_full_rule)

    # (31) t_g <= f_g * M (if cutoff > 0 then f_g=1)
    def t_full_rule(m, g):
        return m.t[g] <= m.f[g] * m.M
    model.t_full = pyo.Constraint(model.G, rule=t_full_rule)

    # Objective (10)
    def objective_rule(m):
        return sum((m.K - m.rank[i,j]) * m.x[i,j] for (i,j) in model.E)
    model.obj = pyo.Objective(rule=objective_rule, sense=pyo.maximize)

    return model


# Solve and output


def solve_model(model, solver_name='cplex'):
    solver = SolverFactory(solver_name)
    result = solver.solve(model, tee=True)
    return result

def print_results(model, name):
    print(f"\n=== {name} ===")
    try:
        obj_value = pyo.value(model.obj)
        print("Objective value:", obj_value)
    except (AttributeError, ValueError):
        print("No feasible solution found or model not solved.")
        return
    
    assigned = [(i,j) for (i,j) in model.E if pyo.value(model.x[i,j]) > 0.5]
    print(f"Number of assigned students: {len(assigned)}")
    
    for g in model.G:
        t_val = pyo.value(model.t[g])
        if t_val is not None:
            print(f"Group {g} cutoff: {t_val:.2f}")
        else:
            print(f"Group {g} cutoff: None")
    
    full_groups = [g for g in model.G if pyo.value(model.f[g]) > 0.5]
    print(f"Full groups: {full_groups}")

if __name__ == "__main__":
    students, colleges, college_cap, college_subject, groups, group_cap, apps = load_strict_data()
    model = build_model(students, colleges, college_cap, college_subject, groups, group_cap, apps)
    solve_model(model)
    print_results(model, "Common quotas (continuous)")


Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer 22.1.0.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2022.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\AmirReza\AppData\Local\Temp\tmptwizs_r0.cplex.log' open.
CPLEX> Problem 'C:\Users\AmirReza\AppData\Local\Temp\tmp6zsjw8rq.pyomo.lp' read.
Read time = 0.08 sec. (5.62 ticks)
CPLEX> Problem name         : C:\Users\AmirReza\AppData\Local\Temp\tmp6zsjw8rq.pyomo.lp
Objective sense      : Maximize
Variables            :   11272  [Box: 24,  Binary: 11248]
Objective nonzeros   :   11184
Linear constraints   :   46328  [Less: 23940,  Greater: 22388]
  Nonzeros           :  244420
  RHS nonzeros       :   46280

Variables            : Min LB: 0.000000         Max UB: 500.0000       
Objective n

In [ ]:
import pandas as pd
import pyomo.environ as pyo
from pyomo.opt import SolverFactory

# Loading the data

def load_strict_data():
    """Load strict dataset (with common quotas)."""
    apps_df = pd.read_csv('strict_applications.csv')
    colleges_df = pd.read_csv('strict_colleges.csv')
    groups_df = pd.read_csv('strict_groups.csv')
    students_df = pd.read_csv('strict_students.csv')

    students = students_df['student_id'].tolist()
    colleges = colleges_df['college_id'].tolist()
    college_cap = colleges_df.set_index('college_id')['capacity'].to_dict()
    college_subject = colleges_df.set_index('college_id')['subject_id'].to_dict()
    groups = groups_df['group_id'].tolist()
    group_cap = groups_df.set_index('group_id')['group_capacity'].to_dict()

    apps = {}
    for _, row in apps_df.iterrows():
        apps[(row['student_id'], row['college_id'])] = {
            'rank': row['rank'],
            'score': row['score'],
            'subject': row['subject_id']
        }

    return students, colleges, college_cap, college_subject, groups, group_cap, apps


# Common quotas – binary cutoff (COM-Irish)


def build_model(students, colleges, college_cap, college_subject, groups, group_cap, apps):
    model = pyo.ConcreteModel()

    # Sets
    model.I = pyo.Set(initialize=students)
    model.J = pyo.Set(initialize=colleges)
    model.E = pyo.Set(initialize=apps.keys())

    # Build groups: subject groups + singleton college groups
    group_colleges = {}
    for g in groups:
        group_colleges[g] = [j for j in colleges if college_subject[j] == g]
    for j in colleges:
        group_colleges[f'col_{j}'] = [j]

    model.G = pyo.Set(initialize=group_colleges.keys())

    group_cap_dict = {}
    for g in group_colleges:
        if isinstance(g, str) and g.startswith('col_'):
            j = int(g.split('_')[1])
            group_cap_dict[g] = college_cap[j]
        else:
            group_cap_dict[g] = group_cap[g]
    model.u_g = pyo.Param(model.G, initialize=group_cap_dict)

    # Score levels per group
    score_levels_g = {}
    for g in model.G:
        scores = sorted({apps[(i,j)]['score'] for (i,j) in apps if j in group_colleges[g]})
        score_levels_g[g] = scores

    # Set of all (group, level) pairs
    model.GK = pyo.Set(initialize=[(g,k) for g in model.G for k in range(1, len(score_levels_g[g])+1)])

    # Number of levels per group (for monotonicity)
    model.K = pyo.Set(model.G, initialize=lambda m, g: range(1, len(score_levels_g[g])+1))

    # Score index for each application and group
    score_idx = {}
    for (i,j) in apps:
        for g in model.G:
            if j in group_colleges[g]:
                s = apps[(i,j)]['score']
                score_idx[(i,j,g)] = score_levels_g[g].index(s) + 1

    # Parameters
    model.rank = pyo.Param(model.E, initialize={(i,j): apps[(i,j)]['rank'] for (i,j) in apps})
    model.score = pyo.Param(model.E, initialize={(i,j): apps[(i,j)]['score'] for (i,j) in apps})
    max_rank = max(model.rank[i,j] for (i,j) in model.E)
    model.K_const = pyo.Param(initialize=max_rank + 1)

    # Variables
    model.x = pyo.Var(model.E, within=pyo.Binary)
    model.t = pyo.Var(model.GK, within=pyo.Binary)

    # (1) Student limit
    def student_limit_rule(m, i):
        return sum(m.x[i,j] for j in model.J if (i,j) in model.E) <= 1
    model.student_limit = pyo.Constraint(model.I, rule=student_limit_rule)

    # (26) Group quotas (common quotas + individual college capacities)
    def group_quota_rule(m, g):
        return sum(m.x[i,j] for (i,j) in apps if j in group_colleges[g]) <= m.u_g[g]
    model.group_quota = pyo.Constraint(model.G, rule=group_quota_rule)

    # (32) Cutoff met: x[i,j] <= t[g, score_idx]
    for (i,j) in apps:
        for g in model.G:
            if j in group_colleges[g]:
                k = score_idx[(i,j,g)]
                setattr(model, f'cutoff_met_{i}_{j}_{g}',
                        pyo.Constraint(expr=model.x[i,j] <= model.t[g, k]))

    # (33) Monotonicity: t[g,k] <= t[g,k+1]
    def monotonicity_rule(m, g, k):
        if k < len(score_levels_g[g]):
            return m.t[g,k] <= m.t[g,k+1]
        return pyo.Constraint.Skip
    model.monotonicity = pyo.Constraint(model.GK, rule=monotonicity_rule)

    # (34) Envy-freeness
    def envy_free_rule(m, i, j):
        better_or_equal = [h for h in model.J if (i,h) in model.E and m.rank[i,h] <= m.rank[i,j]]
        sum_better = sum(m.x[i,h] for h in better_or_equal)
        groups_of_j = [g for g in model.G if j in group_colleges[g]]
        cutoff_terms = []
        for g in groups_of_j:
            k = score_idx[(i,j,g)]
            cutoff_terms.append(1 - m.t[g, k])
        return 1 <= sum_better + sum(cutoff_terms)
    model.envy_free = pyo.Constraint(model.E, rule=envy_free_rule)

    # (35) Non-wastefulness: if cutoff > min score then group is full
    def non_waste_rule(m, g):
        lhs = (1 - m.t[g,1]) * m.u_g[g]
        rhs = sum(m.x[i,j] for (i,j) in apps if j in group_colleges[g])
        return lhs <= rhs
    model.non_waste = pyo.Constraint(model.G, rule=non_waste_rule)

    # Objective (10)
    def objective_rule(m):
        return sum((m.K_const - m.rank[i,j]) * m.x[i,j] for (i,j) in model.E)
    model.obj = pyo.Objective(rule=objective_rule, sense=pyo.maximize)

    return model


# Solve and output


def solve_model(model, solver_name='cplex'):
    solver = SolverFactory(solver_name)
    result = solver.solve(model, tee=True)
    return result

def print_results(model, name):
    print(f"\n=== {name} ===")
    try:
        obj_value = pyo.value(model.obj)
        print("Objective value:", obj_value)
    except (AttributeError, ValueError):
        print("No feasible solution found or model not solved.")
        return
    
    assigned = [(i,j) for (i,j) in model.E if pyo.value(model.x[i,j]) > 0.5]
    print(f"Number of assigned students: {len(assigned)}")

    print("\nGroup cutoffs (lowest accepted score):")
    for g in model.G:
        # Find the lowest score level that is accepted (t=1)
        cutoff_score = None
        for k in model.K[g]:
            if pyo.value(model.t[g,k]) > 0.5:
                pass
        
        cutoff_values = []
        for k in model.K[g]:
            val = pyo.value(model.t[g,k])
            if val is not None:
                cutoff_values.append((k, val))
        
        if cutoff_values:
            lowest_accepted = None
            for k, val in sorted(cutoff_values):
                if val > 0.5:
                    lowest_accepted = k
                    break
            
            if lowest_accepted is not None:
                # We don't have direct access to score_levels_g in the model
                # We can add it as a model attribute, or just print the level number
                print(f"  Group {g}: lowest accepted level = {lowest_accepted}")
            else:
                print(f"  Group {g}: no students accepted")
        else:
            print(f"  Group {g}: no cutoff information")



if __name__ == "__main__":
    students, colleges, college_cap, college_subject, groups, group_cap, apps = load_strict_data()
    model = build_model(students, colleges, college_cap, college_subject, groups, group_cap, apps)
    solve_model(model)
    print_results(model, "Common quotas (binary)")


Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer 22.1.0.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2022.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\AmirReza\AppData\Local\Temp\tmpsl9bo8co.cplex.log' open.
CPLEX> Problem 'C:\Users\AmirReza\AppData\Local\Temp\tmpl3rhnt0m.pyomo.lp' read.
Read time = 0.08 sec. (5.64 ticks)
CPLEX> Problem name         : C:\Users\AmirReza\AppData\Local\Temp\tmpl3rhnt0m.pyomo.lp
Objective sense      : Maximize
Variables            :   27104  [Binary: 27104]
Objective nonzeros   :   11184
Linear constraints   :   50996  [Less: 39812,  Greater: 11184]
  Nonzeros           :  204298
  RHS nonzeros       :   12732

Variables            : Min LB: 0.000000         Max UB: 1.000000       
Objective nonzeros   